In [9]:
import cv2
import urllib.request
import os
import numpy as np

In [11]:
url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
filename = "haarcascade_frontalface_default.xml"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

faceCascade = cv2.CascadeClassifier(filename)

cap = cv2.VideoCapture(0)

last_faces = []

def apply_blue_face(face):
    face = cv2.GaussianBlur(face, (7, 7), 0)

    blue = face.copy()
    blue[:, :, 0] = cv2.add(blue[:, :, 0], 70)
    blue[:, :, 1] = cv2.multiply(blue[:, :, 1], 0.7)
    blue[:, :, 2] = cv2.multiply(blue[:, :, 2], 0.7)

    return blue

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # detect faces every frame (IMPORTANT FIX)
    faces = faceCascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=6,
        minSize=(60, 60)
    )

    if len(faces) > 0:
        last_faces = faces

    for (x, y, w, h) in last_faces:
        
        if w == 0 or h == 0:
            continue

        face_roi = frame[y:y+h, x:x+w]

        if face_roi.size == 0:
            continue

        frame[y:y+h, x:x+w] = apply_blue_face(face_roi)

    cv2.imshow("Stable Blue Face Filter", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()